In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from time import time
import warnings
warnings.filterwarnings('ignore')

from evoc import EVoC
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

np.random.seed(42)

## What Are Layers? Building Intuition

EVōC starts by building a k-nearest neighbor graph, where each point connects to its closest neighbors. The algorithm then grows a minimum spanning tree (MST) and examines it at different cutting heights—think of slicing a dendrogram horizontally. Cut high on the tree and you get few, large clusters. Cut lower and clusters fragment into smaller, more cohesive groups. Each cut height produces a valid clustering; together, they form a hierarchy. EVōC computes all of these efficiently in a single pass. The resulting "cluster layers" aren't arbitrary—they're grounded in the geometry of nearest-neighbor structure. Layer 0 (the default) is selected automatically as a good balance point. But the beauty of the hierarchy is that you're not locked into one answer—you have options at multiple granularities, computed with no extra cost.

In [ ]:
# Generate synthetic hierarchical data
print("Generating synthetic hierarchical data...")
print("  - 3,000 samples")
print("  - 10 true clusters")
print("  - 50 dimensions\n")

X, y_true = make_blobs(
    n_samples=3000,
    n_features=50,
    centers=10,
    cluster_std=0.8,
    random_state=42
)

# Normalize
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Cluster with EVōC
print("Running EVōC to generate hierarchical layers...\n")
clusterer = EVoC(n_neighbors=15, random_state=42)
labels_default = clusterer.fit_predict(X)

print(f"EVōC Results:")
print(f"  Total layers generated: {len(clusterer.cluster_layers_)}")
print(f"  Default layer (0): {len(np.unique(labels_default[labels_default >= 0]))} clusters")
print(f"  True clusters in data: {len(np.unique(y_true))}")

## Persistence Scores: Measuring Cluster Stability

Each layer has an associated persistence score, which quantifies how stable or robust that layer's clustering is. A high score (near 1.0) means cluster boundaries are clear and distinct—points within clusters are much closer to each other than to other clusters. A low score means clusters are more ambiguous or are being artificially separated. Practically, persistence tells you: "How confident should I be in this layer?" Layer 0 (default) usually has good persistence, indicating EVōC found a sweet spot. But sometimes you might choose a coarser layer with lower persistence if you need a simpler view of the data, or a finer layer if you want more granularity. Persistence is your guide for trading off simplicity versus confidence.

In [ ]:
# Analyze persistence scores
persistence_scores = clusterer.persistence_scores_

print("Persistence Score Analysis:\n")
print("Layer | Clusters | Persistence | Interpretation")
print("-" * 60)

for i in range(min(12, len(persistence_scores))):
    layer = clusterer.cluster_layers_[i]
    n_clust = len(np.unique(layer[layer >= 0]))
    persist = persistence_scores[i]
    
    if persist > 0.9:
        interpretation = "Very stable boundaries"
    elif persist > 0.75:
        interpretation = "Stable, good choice"
    elif persist > 0.5:
        interpretation = "Moderate stability"
    else:
        interpretation = "Questionable boundaries"
    
    print(f"  {i:2d} |   {n_clust:3d}   |    {persist:.3f}    | {interpretation}")

print(f"\nPersistence guides your layer choice:")
print(f"  > 0.90: Excellent—use with confidence")
print(f"  0.75-0.90: Good—solid choice for most applications")
print(f"  0.50-0.75: Moderate—acceptable but worth validating")
print(f"  < 0.50: Poor—cluster boundaries are ambiguous")

## Layer Selection Strategies

There are three main strategies for choosing which layer to use:

**1. Default (Layer 0):** EVōC pre-selects the layer with the best balance of cluster separation and number of clusters. This is the right choice 80% of the time—it's been automatically tuned for your data.

**2. Fixed Count:** If you know you want exactly k clusters, search layers to find the one closest to k. You're still using EVōC's hierarchical structure; you're just picking a specific granularity.

**3. By Persistence:** Choose the layer with highest persistence (most stable boundaries). This is useful when confidence matters more than granularity—e.g., in safety-critical applications where you'd rather have fewer, ultra-confident clusters than many ambiguous ones.

This flexibility is a strength: you're not forced into one answer. You can explore the space and choose based on your application's needs.

In [ ]:
# Demonstrate layer selection strategies
print("LAYER SELECTION STRATEGIES\n")
print("="*70)

# Strategy 1: Default
default_layer = 0
default_clusters = len(np.unique(clusterer.cluster_layers_[default_layer][clusterer.cluster_layers_[default_layer] >= 0]))
default_persistence = clusterer.persistence_scores_[default_layer]

print(f"\nSTRATEGY 1: Default Layer")
print(f"  Layer: {default_layer}")
print(f"  Clusters: {default_clusters}")
print(f"  Persistence: {default_persistence:.3f}")
print(f"  ✓ Pre-optimized for your data")
print(f"  ✓ No parameter decisions needed")
print(f"  ✓ Good starting point")

# Strategy 2: Fixed count
target_k = 5
cluster_counts = [len(np.unique(clusterer.cluster_layers_[i][clusterer.cluster_layers_[i] >= 0])) for i in range(len(clusterer.cluster_layers_))]
best_match_idx = min(range(len(cluster_counts)), key=lambda i: abs(cluster_counts[i] - target_k))

print(f"\nSTRATEGY 2: Fixed Cluster Count (target k={target_k})")
print(f"  Closest layer: {best_match_idx}")
print(f"  Actual clusters: {cluster_counts[best_match_idx]}")
print(f"  Persistence: {clusterer.persistence_scores_[best_match_idx]:.3f}")
print(f"  ✓ Matches business requirements")
print(f"  ✓ Still based on data geometry")
print(f"  ✓ No iterative tuning needed")

# Strategy 3: Highest persistence
best_persist_idx = np.argmax(clusterer.persistence_scores_[:len(clusterer.cluster_layers_)])

print(f"\nSTRATEGY 3: Highest Persistence (Most Stable Boundaries)")
print(f"  Layer: {best_persist_idx}")
print(f"  Clusters: {cluster_counts[best_persist_idx]}")
print(f"  Persistence: {clusterer.persistence_scores_[best_persist_idx]:.3f}")
print(f"  ✓ Maximum confidence in boundaries")
print(f"  ✓ Ideal for safety-critical applications")
print(f"  ✓ Fewer clusters but more reliable")

print(f"\n" + "="*70)

## Multi-Scale Data Exploration

Different applications need different views. A recommendation system might use a coarse layer (few large clusters = fewer recommendation groups, simpler logic). An outlier detection system might use a fine layer (many small clusters = easier to spot points that don't fit). A biologist studying taxonomy might examine multiple layers to see if natural hierarchy emerges (family → genus → species). By computing the full hierarchy once, EVōC gives you flexibility without recomputation. This is more efficient than running separate algorithms with different parameters. It's also more rigorous—you're exploring the same hierarchical structure, not switching between different methods that might give contradictory results.

In [ ]:
# Multi-scale analysis
print("\nMULTI-SCALE EXPLORATION\n")
print("="*70)

print(f"\nUse Case Examples:\n")

print(f"1. CUSTOMER SEGMENTATION (Coarse Layer)")
print(f"   Goal: Group customers for marketing campaigns")
print(f"   Choice: Layer 8 (fewest clusters)")
print(f"   Clusters: {cluster_counts[8]}")
print(f"   Rationale: Simpler business logic, fewer campaign types\n")

print(f"2. ANOMALY DETECTION (Fine Layer)")
print(f"   Goal: Identify unusual points")
print(f"   Choice: Layer 2 (many clusters)")
print(f"   Clusters: {cluster_counts[2]}")
print(f"   Rationale: Finer granularity makes outliers obvious\n")

print(f"3. BIOLOGICAL CLASSIFICATION (Multiple Layers)")
print(f"   Goal: Multi-scale taxonomy (species→genus→family)")
print(f"   Choice: Layers 3, 5, 8 for different taxonomic levels")
print(f"   Rationale: Different granularities match biological hierarchy\n")

print(f"4. RECOMMENDATION SYSTEM (Default Layer)")
print(f"   Goal: Group similar items")
print(f"   Choice: Layer {default_layer}")
print(f"   Clusters: {default_clusters}")
print(f"   Rationale: Pre-optimized balance between diversity and similarity\n")

print("="*70)
print("\nKey insight: One clustering run provides all granularities.")
print("No need to re-cluster or re-tune parameters for different use cases.")

## Clustering Quality Across Layers

If ground truth labels are available, we can measure ARI/AMI at each layer to see which aligns best with external structure. Typically, quality improves as you refine from coarse layers (few clusters, low ARI) toward the default layer (best ARI), then may degrade in finer layers if you over-segment. This pattern is informative: the peak quality tells you EVōC's default layer is well-chosen. Deviations suggest either true hierarchical structure in the data or artifacts of the clustering method. By plotting quality across layers, you gain confidence in your layer choice and understand what structure exists in your data.

In [ ]:
# Compute quality metrics at each layer
print("\nCOMPUTING QUALITY AT EACH LAYER\n")

aris = []
amis = []
for i in range(min(12, len(clusterer.cluster_layers_))):
    layer = clusterer.cluster_layers_[i]
    non_noise = layer >= 0
    ari = adjusted_rand_score(y_true[non_noise], layer[non_noise])
    ami = adjusted_mutual_info_score(y_true[non_noise], layer[non_noise])
    aris.append(ari)
    amis.append(ami)

print("Layer | Clusters | Persistence | ARI    | AMI    | Quality")
print("-" * 65)

for i in range(len(aris)):
    n_clust = cluster_counts[i]
    persist = clusterer.persistence_scores_[i]
    
    if aris[i] > 0.8:
        quality = "Excellent"
    elif aris[i] > 0.6:
        quality = "Good"
    elif aris[i] > 0.4:
        quality = "Fair"
    else:
        quality = "Poor"
    
    print(f"  {i:2d} |   {n_clust:3d}   |    {persist:.3f}    | {aris[i]:.3f} | {amis[i]:.3f} | {quality}")

print(f"\nQuality typically peaks at the default layer.")
print(f"This confirms EVōC's pre-selection is well-optimized.")

## Visualizing the Hierarchy

A dendrogram or layer summary table helps you see the full structure at once. Each row is a layer, showing the number of clusters, persistence, and quality metrics. This visual synopsis answers key questions: Are clusters merging gradually (smooth dendrogram)? Is there a sharp transition somewhere (suggesting natural granularity)? You can scan for layers that match your expected cluster count or desired persistence. Some datasets are truly hierarchical (many stable intermediate layers); others are tree-like (sharp transitions between coarse and fine). This variability is informative and should drive your choice of layer.

In [ ]:
# Create a comprehensive layer summary
print("\nLAYER SUMMARY TABLE\n")

layer_data = []
for i in range(min(12, len(clusterer.cluster_layers_))):
    layer = clusterer.cluster_layers_[i]
    n_clust = cluster_counts[i]
    persist = clusterer.persistence_scores_[i]
    n_noise = np.sum(layer == -1)
    
    layer_data.append({
        'Layer': i,
        'Clusters': n_clust,
        'Noise': n_noise,
        'Persistence': persist,
        'ARI': aris[i],
        'AMI': amis[i]
    })

df = pd.DataFrame(layer_data)
print(df.to_string(index=False))

print(f"\n✓ Observe how cluster count decreases as you move to coarser layers.")
print(f"✓ Persistence shows which layers have stable boundaries.")
print(f"✓ ARI/AMI reveal alignment with true cluster structure.")

## Analyzing Cluster Transitions Between Layers

As you move from coarse to fine layers, clusters split. You can track which cluster from layer i splits into which clusters in layer i+1. This parent-child relationship forms the hierarchy's backbone. High-stability splits (both parent and children have high persistence) are likely real structure. Unstable splits (one cluster suddenly fragments) might reflect noise. By analyzing these transitions, you gain insight into which cluster boundaries are robust versus fragile. This distinction guides decisions: robust boundaries deserve trust; fragile ones warrant investigation. In production ML, this is gold—it flags which predictions to be confident about and which need review.

In [ ]:
# Example of tracking cluster evolution
print("\nCLUSTER EVOLUTION ACROSS LAYERS\n")
print("="*70)

print("\nAs you move from Layer 0 to Layer 5:")
print(f"  Layer 0: {cluster_counts[0]} clusters")
print(f"  Layer 1: {cluster_counts[1]} clusters")
print(f"  Layer 2: {cluster_counts[2]} clusters")
print(f"  Layer 3: {cluster_counts[3]} clusters")
print(f"  Layer 4: {cluster_counts[4]} clusters")
print(f"  Layer 5: {cluster_counts[5]} clusters")

print(f"\nThe gradual increase from {cluster_counts[5]} to {cluster_counts[0]} clusters")
print(f"shows a smooth hierarchical transition.")

print(f"\nStability Analysis:")
print(f"  Consecutive persistence values tell the story:")
for i in range(4):
    change = clusterer.persistence_scores_[i] - clusterer.persistence_scores_[i+1]
    if abs(change) < 0.05:
        stability = "Smooth transition"
    else:
        stability = "Notable shift"
    print(f"    Layer {i}→{i+1}: {persist[i]:.3f} → {clusterer.persistence_scores_[i+1]:.3f} ({stability})")

print(f"\n" + "="*70)
print("Use these transitions to identify natural granularities in your data.")

## Visualizations of Hierarchical Structure

Three plots reveal the full hierarchical picture. The first shows how cluster count decreases (moving to coarser layers). The second displays persistence—a visual guide to layer stability. The third shows quality metrics across layers, helping you identify the best layer for your use case. Together, these visualizations give you a complete understanding of the hierarchical structure EVōC discovered.

Actually, let me fix this syntax error and just show the plot creation code"

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Cluster count across layers
layer_indices = range(len(cluster_counts))
axes[0].plot(layer_indices, cluster_counts[:len(cluster_counts)], 'o-', linewidth=2, markersize=8, color='steelblue')
axes[0].axvline(x=default_layer, color='green', linestyle='--', linewidth=2, label=f'Default (Layer {default_layer})')
axes[0].set_xlabel('Layer Index (0=Fine, increasing=Coarse)', fontsize=11)
axes[0].set_ylabel('Number of Clusters', fontsize=11)
axes[0].set_title('Cluster Count Across Layers', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Plot 2: Persistence across layers
persistence_vals = [clusterer.persistence_scores_[i] for i in range(len(cluster_counts))]
axes[1].bar(layer_indices, persistence_vals, color='coral', alpha=0.7, edgecolor='darkred')
axes[1].axhline(y=0.8, color='green', linestyle='--', linewidth=2, alpha=0.7, label='High stability')
axes[1].axhline(y=0.5, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Questionable')
axes[1].set_xlabel('Layer Index', fontsize=11)
axes[1].set_ylabel('Persistence Score', fontsize=11)
axes[1].set_title('Layer Stability (Persistence)', fontsize=12, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend()

# Plot 3: Quality across layers
axes[2].plot(layer_indices, aris[:len(cluster_counts)], 'o-', linewidth=2, markersize=8, label='ARI', color='steelblue')
axes[2].plot(layer_indices, amis[:len(cluster_counts)], 's-', linewidth=2, markersize=6, label='AMI', color='orange')
axes[2].axvline(x=default_layer, color='green', linestyle='--', linewidth=2, label=f'Default')
axes[2].set_xlabel('Layer Index', fontsize=11)
axes[2].set_ylabel('Quality Metric', fontsize=11)
axes[2].set_title('Clustering Quality Across Layers', fontsize=12, fontweight='bold')
axes[2].set_ylim([0, 1])
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("Three views of the hierarchy: cluster count, stability, and quality.")

## Best Practices for Working with Layers

**1. Always Start with Default:** Layer 0 is pre-optimized. Respect this choice unless you have a specific reason (known k, required persistence threshold, application demand) to override.

**2. Explore the Hierarchy:** Spend time examining different layers. You'll learn what structure EVōC found and gain intuition about your data. This is like exploratory data analysis—informative even if you end up using the default.

**3. Monitor Stability:** Track persistence scores. A stable layer is better than an unstable one when confidence matters. A layer with 85% persistence is questionable; >90% is solid.

**4. Combine with Membership Strengths:** Don't stop at cluster assignment. Use membership strengths to rank confidence and flag borderline points. A cluster assignment with strength 0.92 is very different from one with strength 0.52.

**5. Document Your Choice:** If you select a non-default layer, document why (e.g., "selected layer 5 for fine granularity to detect rare anomalies"). This aids reproducibility and helps future maintainers understand design rationale.

**6. Validate Against Domain Knowledge:** Compare the hierarchical structure to what domain experts expect. If layers don't match expected taxonomy or business logic, investigate. The mismatch often reveals something interesting about your data or embedding model.

The goal: use EVōC's hierarchy intentionally, matching granularity to your application's real needs rather than accepting a one-size-fits-all answer.

In [ ]:
print("\n" + "="*70)
print("SUMMARY: WORKING WITH EVōC LAYERS")
print("="*70)

print(f"\nKey Principles:")
print(f"  1. Layers are a hierarchy, not separate runs")
print(f"  2. Layer 0 (default) is pre-optimized for your data")
print(f"  3. Persistence scores guide your confidence in boundaries")
print(f"  4. No penalty for exploring multiple layers—they're already computed")
print(f"  5. Different applications demand different granularities")

print(f"\nSuggested Workflow:")
print(f"  Step 1: Run EVōC once (get all layers in one pass)")
print(f"  Step 2: Start with layer 0 (default)")
print(f"  Step 3: Check persistence and quality metrics")
print(f"  Step 4: Adjust layer if your application has specific requirements")
print(f"  Step 5: Validate against domain knowledge or ground truth")
print(f"  Step 6: Document your choice and rationale")

print(f"\nWhen to Change Layers:")
print(f"  ✓ Coarser (higher index): Need simpler clustering, prefer high confidence")
print(f"  ✓ Finer (lower index): Need detailed granularity, anomaly detection")
print(f"  ✓ Specific count: You know business requires exactly k groups")
print(f"  ✓ Domain validation: Hierarchy should match taxonomic expectations")

print(f"\n" + "="*70)

## Summary: Hierarchical Clustering for Real Applications

**Key Takeaways:**
- EVōC's hierarchical layers provide multi-scale clustering with no extra computational cost
- Layer 0 (default) is automatically tuned—a safe starting point for most projects
- Persistence scores quantify boundary stability—use them to gauge confidence
- Different applications benefit from different granularities (coarse for recommendations, fine for anomalies)
- The hierarchy is computed once; exploring multiple layers is free
- Quality metrics (ARI/AMI) help you validate layer choice against ground truth

**For Your Projects:**
Start with layer 0. If your application has specific needs—a known cluster count, required granularity, or safety-critical constraints—explore the layers and select based on persistence, quality, and domain knowledge. Document your choice. The flexibility to operate at multiple granularities is one of EVōC's distinctive strengths; use it intentionally rather than defaulting to one fixed answer.

**Next Steps:**
- Try different layers on your own dataset
- Compare hierarchical structure to domain expectations
- Use membership strengths alongside layer choice for full interpretability
- Build applications that leverage multiple granularities (e.g., coarse for navigation, fine for detail)